In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import os
import sys

notebook_path = (dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get())
repo_root = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path))

sys.path.insert(0, repo_root)
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

from src.preprocessing.build_dataset import load_modeling_sample
from src.preprocessing.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES
from src.modeling.pipelines import build_model_pipeline
from src.evaluation.validation import split_by_municipality, cross_validate_grouped

spark.sql("USE CATALOG workspace")

sample_df = load_modeling_sample(spark, sample_fraction=0.15)
print(sample_df.shape)

train_df, test_df = split_by_municipality(sample_df)
feature_columns = CATEGORICAL_FEATURES + NUMERIC_FEATURES

for model_name in ["logistic", "hist_gradient_boosting"]:
    pipeline = build_model_pipeline(model_name)
    print(model_name)
    print(cross_validate_grouped(
        pipeline, train_df[feature_columns], train_df["target"], train_df["id_municipio"]
    ).round(3))

In [0]:
model_b_df = sample_df[sample_df["ano"] == 2024].copy()
train_b_df, test_b_df = split_by_municipality(model_b_df)

numeric_b = NUMERIC_FEATURES + ["previous_year_municipal_rate"]
features_b = CATEGORICAL_FEATURES + numeric_b

for model_name in ["logistic", "hist_gradient_boosting"]:
    pipeline = build_model_pipeline(model_name, numeric_features=numeric_b)
    print(model_name)
    print(cross_validate_grouped(
        pipeline, train_b_df[features_b], train_b_df["target"], train_b_df["id_municipio"]
    ).round(3))